# 02 - Stage 2 training

The changeset classifier: five branches, attention fusion. Runs on a Colab T4.

**Before the first real run**, confirm `results/gate1d_framework_spike.json` says PASS.
That gate checks masking actually masks - a model that attends to padding trains fine,
scores fine, and is wrong, and the failure looks like "geometry did not help".

Checkpoint every epoch. Sessions die at 12 h and after 90 min idle, and an unresumable
run is a run you will do twice.

In [ ]:
# Thin by design. Everything real lives in src/mapguard/ - see CONTRIBUTING.md.
!git clone -q https://github.com/ayushmgarg/osm_mapguard.git mapguard 2>/dev/null || (cd mapguard && git pull -q)
%cd mapguard
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, "src")
print(open("results/gate1a_adiff_batch.json").read()[:200])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# Bulk data never goes in the repo. On Colab it goes to Drive so it survives the
# 12 h session limit and the 90 min idle kill.
DATA = Path("/content/drive/MyDrive/mapguard")
DATA.mkdir(parents=True, exist_ok=True)
os.environ["MAPGUARD_DATA"] = str(DATA)
os.environ["MAPGUARD_CONTACT"] = "ayushmgarg10@gmail.com"   # goes in the User-Agent
print("MAPGUARD_DATA =", DATA)

In [ ]:
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## Gate 1d - framework spike

Cheap. Run it here if it has not run locally.

In [ ]:
!python scripts/gate1d_framework_spike.py

## Load the feature store

Built by notebook 01. If you are iterating on the model and not the data, mount Drive and
read the Parquet - do not refetch.

In [ ]:
import os
import pyarrow.parquet as pq

INTERIM = os.environ["MAPGUARD_DATA"] + "/interim"
changesets = pq.read_table(INTERIM + "/changesets.parquet").to_pandas()
edits      = pq.read_table(INTERIM + "/edits.parquet").to_pandas()
user_seq   = pq.read_table(INTERIM + "/user_seq.parquet").to_pandas()
print(len(changesets), "changesets |", len(edits), "sampled edits")
changesets["label"].value_counts(dropna=False)

## Splits

**R1a** is Ovid's protocol - a random split. It is user-leaky and must be labelled as such
in every table it appears in. **R1b** is user-disjoint and is the honest number. Report
both; the gap between them is itself a result.

In [ ]:
from mapguard.evaluation.splits import random_split, user_disjoint_split   # WS2, week 2

## Weights and Biases

Every run records config, split, seed and git SHA. An ablation grid without that is a
spreadsheet of numbers nobody can reproduce.

In [ ]:
import wandb
wandb.login()
run = wandb.init(project="mapguard", config={"split": "R1b", "seed": 20260904})

## Train

Early stopping on validation **PR-AUC**, never accuracy - the classes are imbalanced.

In [ ]:
# from mapguard.training.stage2 import build_model, train      # WS3, week 2-3
# model = build_model(...)
# train(model, ..., checkpoint_dir=os.environ["MAPGUARD_DATA"] + "/checkpoints")

## Results go to results/*.json, not to stdout

In [ ]:
# import json, subprocess
# sha = subprocess.check_output(["git","rev-parse","--short","HEAD"]).decode().strip()
# json.dump({"git_sha": sha, ...}, open("results/stage2_r1b.json","w"), indent=2)